<a href="https://colab.research.google.com/github/ARS-0/Skillify_AI-ML_Internship/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I'm taking help from recorded sessions and AI but i'm learning.

In [ ]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
#Choosing Dataset (Through link bcz it was complex through file in colab)
url = "https://raw.githubusercontent.com/Giskard-AI/examples/main/datasets/twitter_us_airline_sentiment_analysis.csv"
df = pd.read_csv(url)

# columns we actually need
df = df[['text', 'airline_sentiment']]
df.columns = ['text', 'sentiment']

print(df.shape)
df.head()

(1500, 2)


,text,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
negative,886
neutral,342
positive,272


In [ ]:
#PreProcessing Text
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words] # remove stopwords
    return ' '.join(words)

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,said
1,@VirginAmerica plus you've added commercials t...,plus youve added commercials experience tacky
2,@VirginAmerica I didn't today... Must mean I n...,didnt today must mean need take another trip
3,@VirginAmerica it's really aggressive to blast...,really aggressive blast obnoxious entertainmen...
4,@VirginAmerica and it's a really big bad thing...,really big bad thing


In [ ]:
#Splitting into Training and Testing
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 1200
Testing samples: 300


In [ ]:
#Conerting into numbers
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(1200, 3006)


In [ ]:
#Classification Model
# Logistic Regression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_tfidf, y_train)

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

print("Both models trained!")

Both models trained!


In [ ]:
#Don't really understand the functions
for name, model in [("Logistic Regression", log_model), ("Naive Bayes", nb_model)]:
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    print(f"\n{'='*50}")
    print(f"{name} — Accuracy: {acc:.4f}")
    print(f"{'='*50}")
    print(classification_report(y_test, preds))


Logistic Regression — Accuracy: 0.6600
              precision    recall  f1-score   support

    negative       0.67      0.99      0.80       177
     neutral       0.56      0.14      0.23        69
    positive       0.65      0.24      0.35        54

    accuracy                           0.66       300
   macro avg       0.62      0.46      0.46       300
weighted avg       0.64      0.66      0.59       300


Naive Bayes — Accuracy: 0.6033
              precision    recall  f1-score   support

    negative       0.61      1.00      0.76       177
     neutral       0.33      0.01      0.03        69
    positive       0.50      0.06      0.10        54

    accuracy                           0.60       300
   macro avg       0.48      0.36      0.29       300
weighted avg       0.53      0.60      0.47       300



In [ ]:
sample_tweets = [
    "This airline is amazing, best flight ever!",
    "Worst experience, my flight got delayed for 5 hours",
    "The flight was okay, nothing special"
]

sample_clean = [clean_text(t) for t in sample_tweets]
sample_tfidf = vectorizer.transform(sample_clean)

predictions = log_model.predict(sample_tfidf)

for tweet, pred in zip(sample_tweets, predictions):
    print(f"'{tweet}' → {pred}")

'This airline is amazing, best flight ever!' → positive
'Worst experience, my flight got delayed for 5 hours' → negative
'The flight was okay, nothing special' → negative
